In [ ]:
!pip install -U langchain faiss-cpu sentence-transformers huggingface_hub dspy-ai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install accelerate peft transformers datasets -q

In [ ]:
!pip install -U langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain-huggingface -q

In [ ]:
!pip install sentence_transformers -q

In [ ]:
!pip install protobuf -q

In [ ]:
!pip install sentencepiece -q

In [ ]:
!pip install hf_xet

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
CUDA device count: 1
Device name: Tesla T4


## Chunking the text

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Load biomedical passages dataset (passages with passage IDs)
df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

# Clean data
df_passages = df_passages.dropna(subset=["passage"]).drop_duplicates(subset=["passage"])

# Create Documents with metadata = passage ID (as string)
documents = []
for _, row in df_passages.iterrows():
    documents.append(
        Document(
            page_content=row["passage"],
            metadata={"id": str(row.name)}
        )
    )

# Chunk documents (512 tokens per chunk, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
chunked_docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 111480


## Building and Saving FAISS Vector Store

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Use sentence-transformers lightweight embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "all-MiniLM-L6-v2"
print(f"Using device: {device}")

embedding_model = SentenceTransformer(embedding_model_name, device=device)
embedding_wrapper = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={"device": device})

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from tqdm.auto import tqdm

all_embeddings = []
batch_size = 64

for i in tqdm(range(0, len(chunked_docs), batch_size), desc="Embedding batches"):
    batch = chunked_docs[i:i+batch_size]
    texts = [doc.page_content for doc in batch]
    embeddings = embedding_model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    all_embeddings.extend(embeddings)


print(f"Embedded all {len(all_embeddings)} chunks")

# Build FAISS index from embeddings and docs with metadata
text_embedding_pairs = [(doc.page_content, embedding) for doc, embedding in zip(chunked_docs, all_embeddings)]

vectorstore = FAISS.from_embeddings(text_embedding_pairs, embedding_wrapper)
vectorstore.save_local("bio_faiss_index")

Embedding batches:   0%|          | 0/1742 [00:00<?, ?it/s]

Embedded all 111480 chunks


### Langchain

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)

# Load FAISS index with embedding wrapper, allow deserialization
vectorstore = FAISS.load_local(
    "bio_faiss_index",
    embeddings=embedding_wrapper,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="refine",  # or "map_reduce"
    return_source_documents=True
)


Device set to use cuda:0


In [ ]:
OOD_KEYWORDS = ["economy", "inflation", "tariff", "stock market", "war", "climate"]

def is_out_of_domain(query):
    return any(word in query.lower() for word in OOD_KEYWORDS)

def safe_biomedical_qa(query):
    if is_out_of_domain(query):
        return "Sorry, I am only trained to answer biomedical-related questions."
    output = qa_chain.invoke({"query": query})
    return output['result']

In [ ]:
print(safe_biomedical_qa("What is the role of TNF-alpha in inflammation?"))

TNF-alpha is a pleiotropic cytokine that mediates key roles in many physiological and pathological cellular processes including acute and chronic inflammation, programmed cell death or apoptosis, anti-tumor responses, and infection


In [ ]:
print(safe_biomedical_qa("What are the effects of tariffs on the economy?"))

Sorry, I am only trained to answer biomedical-related questions.
